# 05 · Model development I: sentiment classification

**Guideline covered:** Model development

**What this notebook does.** Compares four ways of telling complaints from praise, from simple to advanced, on the same held-out test set:

| Model | Idea | Macro F1, 3 classes | Macro F1, 2 classes |
|---|---|---|---|
| VADER | dictionary of emotional words (no training) | 0.506 | 0.753 |
| Sentence embeddings (English / multilingual MiniLM) + logistic regression | meaning vectors + simple classifier | 0.591 / 0.596 | 0.863 / 0.868 |
| TF-IDF + logistic regression | weighted word counts + simple classifier | 0.601 | 0.877 |
| **Fine-tuned DistilBERT** | pre-trained transformer, trained further on our reviews | **0.643** | **0.896** |

Macro F1 is used because the classes are imbalanced (78% positive). DistilBERT was fine-tuned for 3 epochs with a class-weighted loss; the best epoch on the validation set was kept.

**Is the winner significantly better? (cell X2, McNemar test).** On the same 12,773 test reviews, DistilBERT is right on 1,035 reviews that TF-IDF gets wrong, and wrong on only 204 that TF-IDF gets right (χ² = 556, p ≈ 6 × 10⁻¹²³).

Cell B5 then labels all 85,150 reviews with DistilBERT. It finds **1,667 hidden complaints** (4–5 stars but negative text); stars and model disagree on 9.5% of reviews.

**Inputs:** `model_data.parquet`, `tfidf_X.npz` · **Outputs:** `reviews_with_sentiment.parquet`, `model_results.csv`, `mcnemar_test.csv` · **Hardware:** cells S4 and B1–B5 need a GPU (Colab T4, about 20 minutes per DistilBERT model).

**How to run:** open in Google Colab, keep the project folder at `MyDrive/upi-complaint-analytics` (or change `BASE` in the first code cell) and run the cells from top to bottom. The outputs shown are from the original run.

Phase 6a – Baseline Sentiment Models

Cell S1: load everything and set up the evaluation

In [ ]:
import pandas as pd, numpy as np, scipy.sparse as sp, joblib, torch, os
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

BASE = '/content/drive/MyDrive/upi-complaint-analytics'
PROC = f'{BASE}/data/processed'
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT AVAILABLE - switch runtime!')

data = pd.read_parquet(f'{PROC}/model_data.parquet')
X_tfidf = sp.load_npz(f'{PROC}/tfidf_X.npz')
assert X_tfidf.shape[0] == len(data), 'Row mismatch between data and TF-IDF!'
tr, va, te = [(data.split == s).values for s in ['train', 'val', 'test']]

RESULTS_FILE = f'{BASE}/reports/model_results.csv'
results = pd.read_csv(RESULTS_FILE).to_dict('records') if os.path.exists(RESULTS_FILE) else []

def evaluate(model_name, target, y_true, y_pred):
    labels = ['negative', 'neutral', 'positive'] if target == 'label3' else ['negative', 'positive']
    f1s = f1_score(y_true, y_pred, labels=labels, average=None)
    row = {'model': model_name, 'task': '3-class' if target == 'label3' else '2-class',
           'accuracy': round(accuracy_score(y_true, y_pred), 3), 'macro_f1': round(f1s.mean(), 3),
           **{f'f1_{l}': round(f, 3) for l, f in zip(labels, f1s)}}
    results[:] = [r for r in results if not (r['model'] == model_name and r['task'] == row['task'])]
    results.append(row)
    pd.DataFrame(results).to_csv(RESULTS_FILE, index=False)
    return row

print('Train / val / test:', tr.sum(), va.sum(), te.sum())

GPU: Tesla T4
Train / val / test: 59605 12772 12773


Cell S2: Model 1, VADER (rule-based, no training)

In [ ]:
def vader_3(c):
    return 'positive' if c > 0.05 else ('negative' if c < -0.05 else 'neutral')

for target in ['label3', 'label2']:
    m = te & data[target].notna().values
    scores = data.loc[m, 'vader']
    pred = scores.apply(vader_3) if target == 'label3' else np.where(scores >= 0, 'positive', 'negative')
    print(evaluate('VADER', target, data.loc[m, target], pred))

{'model': 'VADER', 'task': '3-class', 'accuracy': 0.747, 'macro_f1': np.float64(0.506), 'f1_negative': np.float64(0.564), 'f1_neutral': np.float64(0.08), 'f1_positive': np.float64(0.872)}
{'model': 'VADER', 'task': '2-class', 'accuracy': 0.876, 'macro_f1': np.float64(0.753), 'f1_negative': np.float64(0.579), 'f1_positive': np.float64(0.927)}


Cell S3: Model 2, TF-IDF + Logistic Regression (classic machine learning)

In [ ]:
def train_lr(X, model_name, Cs=(0.1, 0.3, 1, 3, 10)):
    models = {}
    for target in ['label3', 'label2']:
        ok = data[target].notna().values
        y = data[target].values
        val_scores = {}
        for C in Cs:                                   # tune on VALIDATION set
            clf = LogisticRegression(C=C, max_iter=3000, class_weight='balanced')
            clf.fit(X[tr & ok], y[tr & ok])
            val_scores[C] = f1_score(y[va & ok], clf.predict(X[va & ok]), average='macro')
        best_C = max(val_scores, key=val_scores.get)
        print(f'{model_name} [{target}] val macro F1 by C:',
              {k: round(v, 3) for k, v in val_scores.items()}, '-> best C =', best_C)
        clf = LogisticRegression(C=best_C, max_iter=3000, class_weight='balanced').fit(X[tr & ok], y[tr & ok])
        print('  TEST:', evaluate(model_name, target, y[te & ok], clf.predict(X[te & ok])))
        models[target] = clf
    return models

lr_tfidf = train_lr(X_tfidf, 'TF-IDF + LogReg')
joblib.dump(lr_tfidf, f'{PROC}/lr_tfidf.joblib')

vocab = np.array(joblib.load(f'{PROC}/tfidf_vectorizer.joblib').get_feature_names_out())
clf = lr_tfidf['label3']
for i, cls in enumerate(clf.classes_):
    print(f'\nWords that most push towards {cls.upper()}:',
          ', '.join(vocab[np.argsort(clf.coef_[i])[::-1][:20]]))

TF-IDF + LogReg [label3] val macro F1 by C: {0.1: 0.602, 0.3: 0.6, 1: 0.598, 3: 0.591, 10: 0.586} -> best C = 0.1
  TEST: {'model': 'TF-IDF + LogReg', 'task': '3-class', 'accuracy': 0.824, 'macro_f1': np.float64(0.601), 'f1_negative': np.float64(0.746), 'f1_neutral': np.float64(0.153), 'f1_positive': np.float64(0.903)}
TF-IDF + LogReg [label2] val macro F1 by C: {0.1: 0.826, 0.3: 0.867, 1: 0.871, 3: 0.871, 10: 0.872} -> best C = 10
  TEST: {'model': 'TF-IDF + LogReg', 'task': '2-class', 'accuracy': 0.921, 'macro_f1': np.float64(0.877), 'f1_negative': np.float64(0.804), 'f1_positive': np.float64(0.95)}

Words that most push towards NEGATIVE: not, bad, useless, fraud, unable, waste, poor, even, otp, fake, bad experience, worst, show, scam, emj_thumbs_down, bed, login, never, wrost, ever

Words that most push towards NEUTRAL: sometimes, not bad, slow, share, average, improve, lag, please, need, thik, jata, screen, emj_wilted_flower, check, good help, overall, not, option, protect, also

W

Cell S4: Model 3, sentence embeddings + Logistic Regression (uses the GPU)

In [ ]:
!pip install -q sentence-transformers
from sentence_transformers import SentenceTransformer

texts = data.text_light.tolist()
EMB = {}
for name, model_id in [('MiniLM-en', 'all-MiniLM-L6-v2'),
                       ('MiniLM-multilingual', 'paraphrase-multilingual-MiniLM-L12-v2')]:
    path = f'{PROC}/emb_{name}.npy'
    if os.path.exists(path):
        EMB[name] = np.load(path)
    else:
        enc = SentenceTransformer(model_id, device='cuda')
        EMB[name] = enc.encode(texts, batch_size=256, show_progress_bar=True, convert_to_numpy=True)
        np.save(path, EMB[name])
    print(name, EMB[name].shape)

for name, E in EMB.items():
    train_lr(E, f'Embeddings {name} + LogReg', Cs=(0.1, 1, 10))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/333 [00:00<?, ?it/s]

MiniLM-multilingual (85150, 384)
Embeddings MiniLM-en + LogReg [label3] val macro F1 by C: {0.1: 0.59, 1: 0.59, 10: 0.587} -> best C = 0.1
  TEST: {'model': 'Embeddings MiniLM-en + LogReg', 'task': '3-class', 'accuracy': 0.808, 'macro_f1': np.float64(0.591), 'f1_negative': np.float64(0.724), 'f1_neutral': np.float64(0.153), 'f1_positive': np.float64(0.897)}
Embeddings MiniLM-en + LogReg [label2] val macro F1 by C: {0.1: 0.858, 1: 0.86, 10: 0.861} -> best C = 10
  TEST: {'model': 'Embeddings MiniLM-en + LogReg', 'task': '2-class', 'accuracy': 0.908, 'macro_f1': np.float64(0.863), 'f1_negative': np.float64(0.784), 'f1_positive': np.float64(0.941)}
Embeddings MiniLM-multilingual + LogReg [label3] val macro F1 by C: {0.1: 0.595, 1: 0.591, 10: 0.592} -> best C = 0.1
  TEST: {'model': 'Embeddings MiniLM-multilingual + LogReg', 'task': '3-class', 'accuracy': 0.811, 'macro_f1': np.float64(0.596), 'f1_negative': np.float64(0.736), 'f1_neutral': np.float64(0.152), 'f1_positive': np.float64(0.899

Cell S5: comparison table so far

In [ ]:
pd.set_option('display.width', 200)
print(pd.DataFrame(results).sort_values(['task', 'macro_f1']).to_string(index=False))

                                  model    task  accuracy  macro_f1  f1_negative  f1_neutral  f1_positive
                                  VADER 2-class     0.876     0.753        0.579         NaN        0.927
          Embeddings MiniLM-en + LogReg 2-class     0.908     0.863        0.784         NaN        0.941
Embeddings MiniLM-multilingual + LogReg 2-class     0.912     0.868        0.792         NaN        0.944
                        TF-IDF + LogReg 2-class     0.921     0.877        0.804         NaN        0.950
                                  VADER 3-class     0.747     0.506        0.564       0.080        0.872
          Embeddings MiniLM-en + LogReg 3-class     0.808     0.591        0.724       0.153        0.897
Embeddings MiniLM-multilingual + LogReg 3-class     0.811     0.596        0.736       0.152        0.899
                        TF-IDF + LogReg 3-class     0.824     0.601        0.746       0.153        0.903


### Cell S3b: wider search for the regularisation strength C
The first grid in S3 put the best C at its edge, so the grid was widened to confirm the optimum (best C = 0.1 for 3 classes).

In [ ]:
lr_tfidf = train_lr(X_tfidf, 'TF-IDF + LogReg', Cs=(0.01, 0.03, 0.1, 0.3, 1, 3, 10, 30, 100))
joblib.dump(lr_tfidf, f'{PROC}/lr_tfidf.joblib')
print(pd.DataFrame(results).sort_values(['task', 'macro_f1']).to_string(index=False))

TF-IDF + LogReg [label3] val macro F1 by C: {0.01: 0.54, 0.03: 0.568, 0.1: 0.602, 0.3: 0.6, 1: 0.598, 3: 0.591, 10: 0.586, 30: 0.582, 100: 0.577} -> best C = 0.1
  TEST: {'model': 'TF-IDF + LogReg', 'task': '3-class', 'accuracy': 0.824, 'macro_f1': np.float64(0.601), 'f1_negative': np.float64(0.746), 'f1_neutral': np.float64(0.153), 'f1_positive': np.float64(0.903)}
TF-IDF + LogReg [label2] val macro F1 by C: {0.01: 0.758, 0.03: 0.797, 0.1: 0.826, 0.3: 0.867, 1: 0.871, 3: 0.871, 10: 0.872, 30: 0.87, 100: 0.862} -> best C = 10
  TEST: {'model': 'TF-IDF + LogReg', 'task': '2-class', 'accuracy': 0.921, 'macro_f1': np.float64(0.877), 'f1_negative': np.float64(0.804), 'f1_positive': np.float64(0.95)}
                                  model    task  accuracy  macro_f1  f1_negative  f1_neutral  f1_positive
                                  VADER 2-class     0.876     0.753        0.579         NaN        0.927
          Embeddings MiniLM-en + LogReg 2-class     0.908     0.863        0.784   

Phase 6b – Fine-tuned DistilBERT

Cell B1: setup (self-contained, so it works even after a restart)

In [ ]:
!pip install -q datasets
import os, numpy as np, pandas as pd, torch
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

BASE = '/content/drive/MyDrive/upi-complaint-analytics'
PROC = f'{BASE}/data/processed'
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT AVAILABLE - switch runtime!')

data = pd.read_parquet(f'{PROC}/model_data.parquet')
RESULTS_FILE = f'{BASE}/reports/model_results.csv'
results = pd.read_csv(RESULTS_FILE).to_dict('records') if os.path.exists(RESULTS_FILE) else []

def evaluate(model_name, target, y_true, y_pred):
    labels = ['negative', 'neutral', 'positive'] if target == 'label3' else ['negative', 'positive']
    f1s = f1_score(y_true, y_pred, labels=labels, average=None)
    row = {'model': model_name, 'task': '3-class' if target == 'label3' else '2-class',
           'accuracy': round(accuracy_score(y_true, y_pred), 3), 'macro_f1': round(f1s.mean(), 3),
           **{f'f1_{l}': round(f, 3) for l, f in zip(labels, f1s)}}
    results[:] = [r for r in results if not (r['model'] == model_name and r['task'] == row['task'])]
    results.append(row)
    pd.DataFrame(results).to_csv(RESULTS_FILE, index=False)
    return row

print(len(data), 'reviews loaded;', len(results), 'earlier results found')

GPU: Tesla T4
85150 reviews loaded; 8 earlier results found


Cell B2: the training function

In [ ]:
import math, inspect
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments,
                          Trainer, DataCollatorWithPadding, set_seed)

MODEL = 'distilbert-base-uncased'
tok = AutoTokenizer.from_pretrained(MODEL)

class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        loss = torch.nn.functional.cross_entropy(
            outputs.logits, labels, weight=self.class_weights.to(outputs.logits.device))
        return (loss, outputs) if return_outputs else loss

def make_args(**kw):
    allowed = inspect.signature(TrainingArguments.__init__).parameters
    dropped = [k for k in kw if k not in allowed]
    if dropped:
        print('Skipped settings not supported by this transformers version:', dropped)
    return TrainingArguments(**{k: v for k, v in kw.items() if k in allowed})

def run_bert(target, epochs=3, batch=32):
    set_seed(42)
    labels = ['negative', 'neutral', 'positive'] if target == 'label3' else ['negative', 'positive']
    l2i = {l: i for i, l in enumerate(labels)}
    d = data[data[target].notna()]

    def to_ds(split):
        part = d[d.split == split]
        ds = Dataset.from_dict({'text': part.text_light.tolist(), 'labels': part[target].map(l2i).tolist()})
        return ds.map(lambda b: tok(b['text'], truncation=True, max_length=128),
                      batched=True, remove_columns=['text'])
    train_ds, val_ds, test_ds = to_ds('train'), to_ds('val'), to_ds('test')

    y_tr = d[d.split == 'train'][target].map(l2i).values
    w = np.sqrt(compute_class_weight('balanced', classes=np.arange(len(labels)), y=y_tr))
    w = torch.tensor(w, dtype=torch.float)
    print(target, 'class weights:', dict(zip(labels, w.numpy().round(2))))

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL, num_labels=len(labels), id2label=dict(enumerate(labels)), label2id=l2i)

    def metrics(p):
        pred = p.predictions.argmax(-1)
        return {'f1_macro': f1_score(p.label_ids, pred, average='macro'),
                'accuracy': accuracy_score(p.label_ids, pred)}

    total_steps = math.ceil(len(train_ds) / batch) * epochs
    args = make_args(
        output_dir=f'/content/bert_{target}', num_train_epochs=epochs, learning_rate=2e-5,
        per_device_train_batch_size=batch, per_device_eval_batch_size=128,
        weight_decay=0.01, warmup_steps=int(0.06 * total_steps),
        eval_strategy='epoch', save_strategy='epoch', save_total_limit=1,
        load_best_model_at_end=True, metric_for_best_model='f1_macro', greater_is_better=True,
        fp16=True, seed=42, report_to='none', logging_steps=200)

    trainer = WeightedTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                              data_collator=DataCollatorWithPadding(tok), compute_metrics=metrics,
                              class_weights=w)
    trainer.train()

    pred = trainer.predict(test_ds).predictions.argmax(-1)
    y_true = [labels[i] for i in test_ds['labels']]
    print('TEST:', evaluate('DistilBERT (fine-tuned)', target, y_true, [labels[i] for i in pred]))

    save_dir = f'{PROC}/distilbert_{target}'
    trainer.save_model(save_dir); tok.save_pretrained(save_dir)
    print('Model saved to', save_dir)
    return trainer, labels

Cell B3: train the 3-class model (~20 min)

In [ ]:
trainer3, labels3 = run_bert('label3')

Map:   0%|          | 0/59605 [00:00<?, ? examples/s]

Map:   0%|          | 0/12772 [00:00<?, ? examples/s]

Map:   0%|          | 0/12773 [00:00<?, ? examples/s]

label3 class weights: {'negative': np.float32(1.34), 'neutral': np.float32(2.94), 'positive': np.float32(0.66)}


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TEST: {'model': 'DistilBERT (fine-tuned)', 'task': '3-class', 'accuracy': 0.889, 'macro_f1': np.float64(0.643), 'f1_negative': np.float64(0.791), 'f1_neutral': np.float64(0.193), 'f1_positive': np.float64(0.944)}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to /content/drive/MyDrive/upi-complaint-analytics/data/processed/distilbert_label3


In [ ]:
print(trainer3.state.best_model_checkpoint)

/content/bert_label3/checkpoint-3726


Cell B4: train the 2-class model (~20 min) and show the full comparison

In [ ]:
trainer2, labels2 = run_bert('label2')
pd.set_option('display.width', 200)
print(pd.DataFrame(results).sort_values(['task', 'macro_f1']).to_string(index=False))

Map:   0%|          | 0/57303 [00:00<?, ? examples/s]

Map:   0%|          | 0/12278 [00:00<?, ? examples/s]

Map:   0%|          | 0/12280 [00:00<?, ? examples/s]

label2 class weights: {'negative': np.float32(1.61), 'positive': np.float32(0.79)}


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TEST: {'model': 'DistilBERT (fine-tuned)', 'task': '2-class', 'accuracy': 0.935, 'macro_f1': np.float64(0.896), 'f1_negative': np.float64(0.832), 'f1_positive': np.float64(0.96)}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to /content/drive/MyDrive/upi-complaint-analytics/data/processed/distilbert_label2
                                  model    task  accuracy  macro_f1  f1_negative  f1_neutral  f1_positive
                                  VADER 2-class     0.876     0.753        0.579         NaN        0.927
          Embeddings MiniLM-en + LogReg 2-class     0.908     0.863        0.784         NaN        0.941
Embeddings MiniLM-multilingual + LogReg 2-class     0.912     0.868        0.792         NaN        0.944
                        TF-IDF + LogReg 2-class     0.921     0.877        0.804         NaN        0.950
                DistilBERT (fine-tuned) 2-class     0.935     0.896        0.832         NaN        0.960
                                  VADER 3-class     0.747     0.506        0.564       0.080        0.872
          Embeddings MiniLM-en + LogReg 3-class     0.808     0.591        0.724       0.153        0.897
Embeddings MiniLM-multilingual + LogReg 3-class     0.811

Cell B5: predict all 85,150 reviews, confusion matrix, error sample

In [ ]:
all_ds = Dataset.from_dict({'text': data.text_light.tolist()}).map(
    lambda b: tok(b['text'], truncation=True, max_length=128), batched=True, remove_columns=['text'])
probs = torch.softmax(torch.tensor(trainer3.predict(all_ds).predictions), dim=-1).numpy()

data['bert_label'] = [labels3[i] for i in probs.argmax(1)]
data['bert_conf'] = probs.max(1).round(3)
for i, l in enumerate(labels3):
    data[f'p_{l}'] = probs[:, i].round(3)
data.to_parquet(f'{PROC}/reviews_with_sentiment.parquet', index=False)
print('Saved reviews_with_sentiment.parquet')

te = data.split == 'test'
cm = pd.DataFrame(confusion_matrix(data.loc[te, 'label3'], data.loc[te, 'bert_label'], labels=labels3),
                  index=[f'true {l}' for l in labels3], columns=[f'pred {l}' for l in labels3])
print('\nConfusion matrix (test set):\n', cm)
print('\nMacro F1 by app (test set):\n',
      data[te].groupby('app').apply(lambda g: f1_score(g.label3, g.bert_label, average='macro')).round(3))
print('\nShare of all reviews where stars and model disagree:', round((data.label3 != data.bert_label).mean(), 3))

pd.set_option('display.max_colwidth', 150)
err = data[te & (data.label3 != data.bert_label) & data.is_long].sample(25, random_state=1)
print(err[['app', 'rating', 'bert_label', 'bert_conf', 'review_text']].to_string())

Map:   0%|          | 0/85150 [00:00<?, ? examples/s]

Saved reviews_with_sentiment.parquet

Confusion matrix (test set):
                pred negative  pred neutral  pred positive
true negative           1832           169            365
true neutral             135            90            268
true positive            301           181           9432

Macro F1 by app (test set):
 app
BHIM          0.654
Google Pay    0.626
Paytm         0.612
PhonePe       0.595
dtype: float64

Share of all reviews where stars and model disagree: 0.095
              app  rating bert_label  bert_conf                                                                                                                                                                                                                                                                                                                                                                                                                review_text
37711        BHIM       5    neutral      0.662     

/tmp/ipykernel_1555/1178867018.py:17: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  data[te].groupby('app').apply(lambda g: f1_score(g.label3, g.bert_label, average='macro')).round(3))


## Is the best model significantly better? (cell X2, McNemar test)
Compares DistilBERT and TF-IDF + logistic regression on the same test reviews, looking only at reviews where exactly one of the two is right (Dietterich, 1998).

In [12]:
# Cell X2: is fine-tuned DistilBERT significantly better than TF-IDF + Logistic Regression on the SAME test reviews?
# McNemar test: looks only at reviews where exactly one of the two models is right (paired comparison).
import os, joblib, numpy as np, pandas as pd, scipy.sparse as sp
from sklearn.metrics import f1_score
from statsmodels.stats.contingency_tables import mcnemar

BASE = '/content/drive/MyDrive/upi-complaint-analytics'
PROC, REP, DASH = f'{BASE}/data/processed', f'{BASE}/reports', f'{BASE}/dashboard'
data = pd.read_parquet(f'{PROC}/model_data.parquet')
X = sp.load_npz(f'{PROC}/tfidf_X.npz')
lr = joblib.load(f'{PROC}/lr_tfidf.joblib')
if isinstance(lr, dict):                      # file holds several models: take the 3-class one
    print('Saved models:', {k: type(v).__name__ for k, v in lr.items()})
    lr = next(v for v in lr.values() if hasattr(v, 'predict') and len(getattr(v, 'classes_', [])) == 3)
bert = pd.read_parquet(f'{PROC}/reviews_with_sentiment.parquet')[['review_id', 'bert_label']]
assert X.shape[0] == len(data)
te = (data.split == 'test').values
y = data.label3.values[te]
p_lr = lr.predict(X[te])
p_bert = data[['review_id']].merge(bert, on='review_id', how='left').bert_label.values[te]
print('TF-IDF model classes:', list(lr.classes_))
f1_lr = f1_score(y, p_lr, average='macro')
f1_bert = f1_score(y, p_bert, average='macro')
print(f'Test macro F1 (3 classes) - TF-IDF + LogReg: {f1_lr:.3f} | DistilBERT: {f1_bert:.3f}   (should match model_results.csv)')

a, b = p_lr == y, p_bert == y
table = [[int((a & b).sum()), int((a & ~b).sum())], [int((~a & b).sum()), int((~a & ~b).sum())]]
res = mcnemar(table, exact=False, correction=True)
print('\n                     DistilBERT right   DistilBERT wrong')
print(f'TF-IDF right        {table[0][0]:>16,} {table[0][1]:>18,}')
print(f'TF-IDF wrong        {table[1][0]:>16,} {table[1][1]:>18,}')
print(f'\nMcNemar chi-square = {res.statistic:.1f}, p = {res.pvalue:.1e}')
print(f'DistilBERT fixes {table[1][0]:,} test reviews that TF-IDF gets wrong, and breaks {table[0][1]:,} that TF-IDF gets right.')

mc = pd.DataFrame([{'comparison': 'DistilBERT vs TF-IDF + LogReg (3 classes, test set)', 'test_reviews': int(te.sum()),
                    'both_right': table[0][0], 'only_tfidf_right': table[0][1], 'only_distilbert_right': table[1][0],
                    'both_wrong': table[1][1], 'macro_f1_tfidf': round(f1_lr, 3), 'macro_f1_distilbert': round(f1_bert, 3),
                    'mcnemar_chi2': round(res.statistic, 1), 'p_value': res.pvalue}])
mc.to_csv(f'{REP}/mcnemar_test.csv', index=False)
row = {'test': 'McNemar', 'where': 'Cell X2', 'question': 'Is fine-tuned DistilBERT significantly better than TF-IDF + LogReg on the same test reviews?',
       'result': f"Yes: right on {table[1][0]:,} reviews TF-IDF gets wrong vs {table[0][1]:,} the other way; p = {res.pvalue:.1e}"
                 if res.pvalue < 0.05 else f"No significant difference (p = {res.pvalue:.2f})"}
for folder in [REP, DASH]:
    f = f'{folder}/statistical_tests_summary.csv'
    s = pd.read_csv(f)
    s = pd.concat([s[s.test != 'McNemar'], pd.DataFrame([row])], ignore_index=True)
    s.to_csv(f, index=False)
print('\nSaved reports/mcnemar_test.csv and added the test to statistical_tests_summary.csv')

Saved models: {'label3': 'LogisticRegression', 'label2': 'LogisticRegression'}
TF-IDF model classes: ['negative', 'neutral', 'positive']
Test macro F1 (3 classes) - TF-IDF + LogReg: 0.601 | DistilBERT: 0.643   (should match model_results.csv)

                     DistilBERT right   DistilBERT wrong
TF-IDF right                  10,319                204
TF-IDF wrong                   1,035              1,215

McNemar chi-square = 556.0, p = 6.2e-123
DistilBERT fixes 1,035 test reviews that TF-IDF gets wrong, and breaks 204 that TF-IDF gets right.

Saved reports/mcnemar_test.csv and added the test to statistical_tests_summary.csv
